### LEVEL 5 - AGENTIC AI WITH TOOLS

In [1]:
import pandas as pd
import json
import os
from datetime import datetime

print("✅ Level 5 - Agentic AI with Tools Initialized")

✅ Level 5 - Agentic AI with Tools Initialized


##### 2: Load Data & Memory

In [2]:
# Load the latest dataset
df = pd.read_excel('C:/Users/HP/Projects/Gen AI Powered Data Analytics/Final_Dataset_with_Agent_Level4.xlsx')

# Load memory
memory_file = 'logs/customer_memory_level4.json'
if os.path.exists(memory_file):
    with open(memory_file, 'r') as f:
        customer_memory = json.load(f)
    print(f"✅ Loaded memory for {len(customer_memory)} customers")
else:
    customer_memory = {}
    print("ℹ️ Starting with fresh memory")

print(f"Total customers: {len(df)}")

✅ Loaded memory for 500 customers
Total customers: 500


In [7]:
# Fix Block: Add llm_powered_reasoning function
def llm_powered_reasoning(customer_row, memory):
    """Simulated LLM Reasoning for Level 5"""
    cust_id = customer_row['Customer_ID']
    history = memory.get(cust_id, {"interactions": []})
    prev_count = len(history["interactions"])
    
    risk_score = calculate_risk_score(customer_row)
    
    if risk_score >= 80:
        decision = "ESCALATE"
        reason = "Critical risk level detected. High missed payments and utilization require immediate human intervention to prevent default."
    elif risk_score >= 60:
        decision = "PAYMENT_PLAN"
        reason = "High risk but recoverable. A structured payment plan combined with support would be the most effective and empathetic approach."
    elif risk_score >= 35:
        decision = "SMS_REMINDER"
        reason = "Moderate risk. A timely, personalized reminder may resolve the issue before it escalates."
    else:
        decision = "MONITOR"
        reason = "Low immediate risk. Best to continue passive monitoring while watching for any changes."
    
    # Save to memory
    history["interactions"].append({
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "risk_score": risk_score,
        "action": decision,
        "reason": reason
    })
    
    memory[cust_id] = history
    
    return decision, reason, risk_score

In [9]:
# === MISSING FUNCTION - Add this block first ===
def calculate_risk_score(row):
    """Calculate risk score (0-100) based on key features"""
    score = 0
    
    # Missed Payments - strongest factor
    if row['Missed_Payments'] >= 4:
        score += 50
    elif row['Missed_Payments'] >= 2:
        score += 25
    
    # Credit Utilization
    if row['Credit_Utilization'] > 0.70:
        score += 30
    elif row['Credit_Utilization'] > 0.50:
        score += 15
    
    # Credit Score (lower = higher risk)
    if row['Credit_Score'] < 500:
        score += 15
    elif row['Credit_Score'] < 600:
        score += 8
    
    return min(score, 100)

In [11]:
# Code Block 4: Guardrails (Same as before but cleaner)
def apply_guardrails_level4(customer_row, decision):
    age = customer_row.get('Age', 30)
    employment = str(customer_row.get('Employment_Status', '')).lower()
    
    if age >= 65 or 'retired' in employment or 'unemployed' in employment:
        if decision == "ESCALATE":
            return "PAYMENT_PLAN", "Softened action for vulnerable customer"
    
    if customer_row.get('Missed_Payments', 0) <= 1 and decision in ["ESCALATE", "PAYMENT_PLAN"]:
        return "SMS_REMINDER", "First-time mild case - softened action"
    
    return decision, "No guardrail triggered"

# Main Level 4 Agent
def run_level4_agent(customer_row):
    cust_id = customer_row['Customer_ID']
    
    decision, reason, risk_score, context = llm_powered_reasoning(customer_row, customer_memory)
    final_action, guardrail_reason = apply_guardrails_level4(customer_row, decision)
    
    log = {
        'Customer_ID': cust_id,
        'Risk_Score': risk_score,
        'Initial_Decision': decision,
        'Final_Action': final_action,
        'Reason': reason,
        'Guardrail': guardrail_reason,
        'Timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    
    return log, final_action

##### 3: Define the Tools (The Actions the Agent Can Take)

In [3]:
# Code Block 3: Tool Definitions
def send_sms_reminder(customer_row):
    """Tool 1: Send a gentle SMS reminder"""
    print(f"📱 SMS sent to {customer_row['Customer_ID']}: 'Hi, we noticed a missed payment. Please pay at your earliest convenience.'")
    return "SMS_SENT"

def offer_payment_plan(customer_row):
    """Tool 2: Offer a structured payment plan"""
    print(f"📧 Payment plan offered to {customer_row['Customer_ID']}: Flexible installments suggested.")
    return "PAYMENT_PLAN_OFFERED"

def escalate_to_human(customer_row):
    """Tool 3: Escalate to human collections officer"""
    print(f"👤 Escalated {customer_row['Customer_ID']} to human collections team for immediate follow-up.")
    return "ESCALATED_TO_HUMAN"

def update_customer_status(customer_row, new_status):
    """Tool 4: Update customer record"""
    print(f"📊 Updated status for {customer_row['Customer_ID']} to: {new_status}")
    return "STATUS_UPDATED"

def log_action(customer_row, action_taken):
    """Tool 5: Log the action taken"""
    print(f"📝 Logged action '{action_taken}' for customer {customer_row['Customer_ID']}")
    return "LOGGED"

##### 4: Tool Registry & Tool Selection Logic

In [4]:
# Code Block 4: Tool Registry and Selection
tools = {
    "SMS_REMINDER": send_sms_reminder,
    "PAYMENT_PLAN": offer_payment_plan,
    "ESCALATE": escalate_to_human,
    "MONITOR": lambda x: "MONITORING_CONTINUED"   # No action needed
}

def select_and_execute_tool(customer_row, decision):
    """
    Selects the appropriate tool based on the decision and executes it.
    """
    tool_function = tools.get(decision, log_action)
    
    if decision == "MONITOR":
        result = "MONITORING_CONTINUED"
    elif decision == "ESCALATE":
        result = tool_function(customer_row)
    elif decision == "PAYMENT_PLAN":
        result = tool_function(customer_row)
    else:  # SMS_REMINDER
        result = tool_function(customer_row)
    
    # Log the action
    log_action(customer_row, decision)
    
    return result

##### 5: Updated Main Agent with Tools

In [5]:
# Code Block 5: Main Agent Runner with Tools
def run_agent_with_tools(customer_row):
    """
    Full Agent: Reasoning → Decision → Tool Execution
    """
    cust_id = customer_row['Customer_ID']
    
    # Step 1: Get decision from reasoning (using our simulated LLM)
    decision, reason, risk_score = llm_powered_reasoning(customer_row, customer_memory)
    
    # Step 2: Apply guardrails
    final_action, guardrail_reason = apply_guardrails_level4(customer_row, decision)
    
    # Step 3: Execute the Tool
    tool_result = select_and_execute_tool(customer_row, final_action)
    
    # Step 4: Create detailed log
    log = {
        'Customer_ID': cust_id,
        'Risk_Score': risk_score,
        'Initial_Decision': decision,
        'Final_Action': final_action,
        'Reason': reason,
        'Guardrail_Applied': guardrail_reason,
        'Tool_Result': tool_result,
        'Timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    
    return log, final_action

##### 6: Test the Agent with Tools

In [12]:
# Code Block 6: Test Agent with Tools
print("🚀 Testing Level 5 Agent with Tools\n")

test_customers = df.sample(5, random_state=42)

for idx, customer in test_customers.iterrows():
    log, final_action = run_agent_with_tools(customer)
    
    print(f"Customer     : {customer['Customer_ID']}")
    print(f"Risk Score   : {log['Risk_Score']}")
    print(f"Decision     : {log['Initial_Decision']} → {final_action}")
    print(f"Reason       : {log['Reason'][:120]}...")
    print(f"Tool Result  : {log['Tool_Result']}")
    print("-" * 85)

🚀 Testing Level 5 Agent with Tools

📱 SMS sent to CUST0362: 'Hi, we noticed a missed payment. Please pay at your earliest convenience.'
📝 Logged action 'SMS_REMINDER' for customer CUST0362
Customer     : CUST0362
Risk Score   : 55
Decision     : SMS_REMINDER → SMS_REMINDER
Reason       : Moderate risk. A timely, personalized reminder may resolve the issue before it escalates....
Tool Result  : SMS_SENT
-------------------------------------------------------------------------------------
📧 Payment plan offered to CUST0074: Flexible installments suggested.
📝 Logged action 'PAYMENT_PLAN' for customer CUST0074
Customer     : CUST0074
Risk Score   : 70
Decision     : PAYMENT_PLAN → PAYMENT_PLAN
Reason       : High risk but recoverable. A structured payment plan combined with support would be the most effective and empathetic ap...
Tool Result  : PAYMENT_PLAN_OFFERED
-------------------------------------------------------------------------------------
📝 Logged action 'MONITOR' for customer C

##### 7: Final Summary & Save

In [15]:
# Code Block 7: Final Summary and Save
print("="*85)
print("LEVEL 5 - AGENTIC AI WITH TOOLS - FINAL SUMMARY")
print("="*85)
print(f"Total customers processed : {len(df)}")
print(f"Customers with memory     : {len(customer_memory)}")

# Action Distribution
action_counts = pd.Series([log.get('Final_Action', 'UNKNOWN') 
                          for log in [run_agent_with_tools(row)[0] for _, row in df.iterrows()][:50]]).value_counts()

print("\nAction Distribution (Sample):")
for action, count in action_counts.items():
    print(f"   {action:15} : {count} customers")

# Save everything
os.makedirs('logs', exist_ok=True)

with open('logs/customer_memory_with_tools.json', 'w') as f:
    json.dump(customer_memory, f, indent=2)

df.to_excel('C:/Users/HP/Projects/Gen AI Powered Data Analytics/Final_Dataset_with_Agent_Level5_Tools.xlsx', index=False)

print("\n✅ Level 5 Completed Successfully!")
print("📁 Files saved:")
print("   • logs/customer_memory_with_tools.json")
print("   • data/processed/Final_Dataset_with_Agent_Level5_Tools.xlsx")
print("\n🎉 I now have a true Agentic AI with Tools!")
print("Features:")
print("   • Natural reasoning")
print("   • Memory")
print("   • Ethical guardrails")
print("   • Tool execution (SMS, Payment Plan, Escalate, etc.)")

LEVEL 5 - AGENTIC AI WITH TOOLS - FINAL SUMMARY
Total customers processed : 500
Customers with memory     : 500
📱 SMS sent to CUST0001: 'Hi, we noticed a missed payment. Please pay at your earliest convenience.'
📝 Logged action 'SMS_REMINDER' for customer CUST0001
📧 Payment plan offered to CUST0002: Flexible installments suggested.
📝 Logged action 'PAYMENT_PLAN' for customer CUST0002
📝 Logged action 'MONITOR' for customer CUST0003
📱 SMS sent to CUST0004: 'Hi, we noticed a missed payment. Please pay at your earliest convenience.'
📝 Logged action 'SMS_REMINDER' for customer CUST0004
📱 SMS sent to CUST0005: 'Hi, we noticed a missed payment. Please pay at your earliest convenience.'
📝 Logged action 'SMS_REMINDER' for customer CUST0005
📧 Payment plan offered to CUST0006: Flexible installments suggested.
📝 Logged action 'PAYMENT_PLAN' for customer CUST0006
📱 SMS sent to CUST0007: 'Hi, we noticed a missed payment. Please pay at your earliest convenience.'
📝 Logged action 'SMS_REMINDER' for cu